# Word Embeddings and Recurrent Neural Networks

**Deadline:** May 19, 2026
**Total:** 10 points

# 🎯 Objective

The goal of this assignment is to connect the lecture on **Word2Vec, GloVe, vanilla RNNs, LSTMs, and GRUs** with small practical experiments.

You will work with four related ideas:


1. pretrained word embeddings and word-vector arithmetic,
2. the full softmax loss used in Word2Vec Skip-gram,
3. a manually implemented vanilla RNN forward pass using PyTorch autograd,
4. an empirical comparison of LSTM and GRU models for text classification.

The key message is:

$$
\text{embeddings turn words into vectors; recurrent models process sequences of those vectors.}
$$


---

# 📌 General Requirements

* Use **Python** and **PyTorch**.
* Submit a **Jupyter Notebook (.ipynb)**.
* The notebook must:
  * run from top to bottom without errors,
  * include short markdown explanations,
  * include printed tensor shapes,
  * include at least one table or plot for Task 4,
  * set a random seed,
  * clearly separate the four tasks.

You may use GPU if available. CPU is acceptable if you use small subsets and small models.


---

# ⚠ Restrictions

✅ Allowed:

* `torch`, `torch.nn`, `torch.optim`
* `torchtext` or another simple source for text data / pretrained embeddings
* `gensim` for loading pretrained GloVe or Word2Vec vectors
* `numpy`, `pandas`, `matplotlib`
* `sklearn` only for simple train / validation splitting or metrics

❌ Not allowed:

* high-level training frameworks such as Lightning, fastai, or Trainer APIs,
* submitting copied tutorial code without explanation,
* using an RNN / LSTM / GRU as a black box without inspecting shapes,


---

# 📚 Recommended Reading

Before starting, review:

* [Word2Vec](/doc/82dcb013-4327-4a83-b129-b83cfd2b4302) 
* [GloVe: Global Vectors for Word Representation](/doc/dcd63672-2aa9-4588-8deb-3d3916fe3422) 
* [LSTM and GRU](/doc/76aed288-947b-4c4b-94a0-af908413ef98) 
* PyTorch documentation for `nn.Embedding`, `nn.RNN`, `nn.LSTM`, and `nn.GRU`
* TorchText documentation, if you choose to use TorchText datasets or GloVe vectors


---






# Task 1 — Pretrained Word Embeddings and Word Algebra (2 pts)

Use pretrained word embeddings such as **GloVe** or **Word2Vec** and explore whether vector arithmetic captures semantic relationships.

Examples of classical word-vector algebra:

```text
king - man + woman ≈ queen
paris - france + italy ≈ rome
walking - walk + swim ≈ swimming
```

## Requirements


1. Load pretrained word vectors.
   * Recommended: `glove-wiki-gigaword-50` from `gensim.downloader`, or GloVe vectors from `torchtext.vocab`.
2. Print:
   * embedding dimension,
   * vocabulary size, if available,
   * vector shape for at least one selected word.
3. Choose at least **8 words** and compute cosine similarities between selected pairs.
4. Perform at least **5 word analogies** using vector arithmetic.
5. For each analogy, show the top 5 nearest words.
6. Create a small table:

| Analogy | Expected answer | Top result | Was it reasonable? | Short comment |
|---------|-----------------|------------|--------------------|---------------|
| king - man + woman | queen           | ...        | yes/no             | ...           |


7. Include at least one failed or imperfect example and explain why pretrained embeddings may fail.

## Required discussion

Briefly explain:

* what a word embedding represents,
* why cosine similarity is useful for word vectors,
* why vector arithmetic can sometimes encode semantic relationships,
* why it does not always work,


---

In [20]:
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [21]:
import gensim.downloader as api

vectors = api.load("glove-wiki-gigaword-50")

print("Vector size:", vectors.vector_size)
print("Vector for 'king':", vectors["king"].shape)

# Cosine similarity
print(vectors.similarity("cat", "dog"))
print(vectors.similarity("cat", "car"))


test = [
    {'positive': ["king", "woman"], 'negative': ["man"]},
    {'positive': ["rome", "germany"], 'negative': ["italy"]},
    {'positive': ["car", "water"], 'negative': ["land"]},
    {'positive': ["watermelon", "vegetable"], 'negative': ["fruit"]},
    {'positive': ["orange", "vegetable"], 'negative': ["fruit"]},
]

for item in test:
    pos = item['positive']
    neg = item['negative']
    result = vectors.most_similar(
        positive=pos,
        negative=neg,
        topn=5,
    )
    print(f"Positive: {pos}, Negative: {neg}")
    print(result)


Vector size: 50
Vector for 'king': (50,)
0.9218005
0.36382526
Positive: ['king', 'woman'], Negative: ['man']
[('queen', 0.8523604273796082), ('throne', 0.7664334177970886), ('prince', 0.7592144012451172), ('daughter', 0.7473883628845215), ('elizabeth', 0.7460219860076904)]
Positive: ['rome', 'germany'], Negative: ['italy']
[('berlin', 0.8804532885551453), ('vienna', 0.8153893351554871), ('warsaw', 0.8132875561714172), ('munich', 0.7752045392990112), ('moscow', 0.7687922716140747)]
Positive: ['car', 'water'], Negative: ['land']
[('truck', 0.7373852729797363), ('cars', 0.7103557586669922), ('exhaust', 0.7027417421340942), ('tires', 0.6924084424972534), ('smoke', 0.6907373666763306)]
Positive: ['watermelon', 'vegetable'], Negative: ['fruit']
[('tomato', 0.7323246598243713), ('tofu', 0.7162290811538696), ('guacamole', 0.708424985408783), ('tortilla', 0.7047438621520996), ('avocado', 0.702354371547699)]
Positive: ['orange', 'vegetable'], Negative: ['fruit']
[('cream', 0.6644747853279114), (

# Task 2 — Full Softmax Loss for Skip-gram Word2Vec (2 pts)

In this task, you will manually calculate the **full softmax loss** for a tiny Word2Vec Skip-gram example.

You should not use negative sampling here. The goal is to see exactly why full softmax requires scores for every word in the vocabulary.

## Tiny corpus

Use a very small corpus, for example:

```python
corpus = ["the", "cat", "sat", "on", "the", "mat"]
window_size = 1
```

From this corpus, create Skip-gram center-context pairs.

For `window_size = 1`, the word `cat` produces:

```text
center = cat, outside = the
center = cat, outside = sat
```

## Requirements


1. Build the vocabulary and word-to-index dictionary.
2. Generate all Skip-gram training pairs for the tiny corpus.
3. Create small input and output embedding matrices manually or with a fixed random seed:

```python
V = len(vocab)
d = 3
input_embeddings = torch.randn(V, d)
output_embeddings = torch.randn(V, d)
```


4. For one selected pair `(center_word, outside_word)`, compute manually:
   * center vector $v_c$,
   * all scores $s_k = u_k^T v_c$,
   * softmax probabilities over the full vocabulary,
   * negative log-likelihood loss $-\log P(outside \mid center)$.
5. Verify the result using `torch.nn.functional.cross_entropy`.
6. Compute the average full softmax loss over all Skip-gram pairs.
7. Print intermediate values clearly for the selected pair.

## Required table

For one selected pair, create a table like this:

| Vocabulary word | Score | Softmax probability | Target |
|-----------------|------:|--------------------:|-------:|
| the             | ...   | ...                 | 1 or 0 |
| cat             | ...   | ...                 | 1 or 0 |
| ...             | ...   | ...                 | ...    |

## Required discussion

Briefly explain:

* what the center word and outside word mean,
* why the denominator of the softmax uses all vocabulary words,
* why this is expensive for large vocabularies,
* why negative sampling is often used in practice,
* what PyTorch autograd would compute if `requires_grad=True`.



In [55]:
corpus = ["the", "cat", "sat", "on", "the", "mat"]
window_size = 1

vocab = list(dict.fromkeys(corpus))
word_to_index = {word: idx for idx, word in enumerate(vocab)}
index_to_word = {idx: word for word, idx in word_to_index.items()}

print(word_to_index)

pairs = []

# Generate (center_word, context_word) pairs
for idx, center_word in enumerate(corpus):
    for offset in range(-window_size, window_size + 1):
        if offset == 0:
            continue
        context_idx = idx + offset
        if 0 <= context_idx < len(corpus):
            pairs.append((word_to_index[center_word], word_to_index[corpus[context_idx]]))

print(pairs)
    

g = torch.Generator().manual_seed(42)

V = len(vocab)
d = 3
input_embeddings = torch.randn(V, d, generator=g)
output_embeddings = torch.randn(V, d, generator=g)

print((input_embeddings))
print((output_embeddings))
print("Input embeddings shape:", input_embeddings.shape)
print("Output embeddings shape:", output_embeddings.shape)


center_word = "cat"
outside_word = "the"

selected_pair = (
    word_to_index[center_word],
    word_to_index[outside_word]
)

v_c = input_embeddings[selected_pair[0]]

scores = torch.matmul(output_embeddings, v_c)
print(f"Scores {scores}")

probs = torch.softmax(scores, dim=0)
print(f"Probabilities {probs}")

loss = -torch.log(probs[selected_pair[1]])

print(f"Loss: {loss.item()}")

{'the': 0, 'cat': 1, 'sat': 2, 'on': 3, 'mat': 4}
[(0, 1), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2), (3, 0), (0, 3), (0, 4), (4, 0)]
tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863],
        [ 2.2082, -0.6380,  0.4617],
        [ 0.2674,  0.5349,  0.8094],
        [ 1.1103, -1.6898, -0.9890]])
tensor([[ 0.9580,  1.3221,  0.8172],
        [-0.7658, -0.7506,  1.3525],
        [ 0.6863, -0.3278,  0.7950],
        [ 0.2815,  0.0562,  0.5227],
        [-0.2384, -0.0499,  0.5263]])
Input embeddings shape: torch.Size([5, 3])
Output embeddings shape: torch.Size([5, 3])
Scores tensor([-1.4162,  0.4144,  0.3780, -0.0956, -0.0969])
Probabilities tensor([0.0482, 0.3008, 0.2900, 0.1806, 0.1804])
Loss: 3.0319786071777344


# Task 3 — Manual Vanilla RNN Forward Pass with Pretrained Embeddings (3 pts)

Implement a simple vanilla RNN text classifier. You may use PyTorch autograd for gradients, but you must manually implement the recurrent forward pass instead of using `nn.RNN`.

The recurrent update should be:

$$
h_t = \tanh(x_t W_{xh} + h_{t-1} W_{hh} + b_h)
$$

The classifier should use the final hidden state:

$$
\text{logits} = h_T W_{hy} + b_y
$$

## Dataset

Use a small text classification dataset. Recommended options:

* `torchtext.datasets.AG_NEWS`, using a small subset, or
* a small custom dataset of short sentences with labels, if TorchText causes installation problems.

For AG_NEWS, you may use only a subset such as **2,000 training** **examples** and **500 validation** examples to keep the assignment fast.

## Requirements


1. Load or create a text classification dataset.
2. Tokenize text using a simple tokenizer, for example lowercase + `.split()`.
3. Build a vocabulary.
4. Convert each text into a sequence of token IDs.
5. Pad or truncate sequences to a fixed length.
6. Load **pretrained embeddings** when possible.
   * If a token is found in the pretrained vectors, initialize its embedding from the pretrained vector.
   * If it is missing, initialize randomly.
7. Implement a custom `ManualRNNClassifier(nn.Module)`.
8. In `forward`, explicitly loop over time steps:

```python
for t in range(seq_len):
    x_t = embedded[:, t, :]
    h = torch.tanh( ___TODO__ )
```


 9. Train for at least **2 epochs**.
10. Print:
    * input batch shape,
    * embedded batch shape,
    * hidden state shape,
    * logits shape.
11. Report validation accuracy.

## Required discussion

Briefly explain:

* why the embedding layer converts token IDs into vectors,
* how the hidden state changes over time,
* why the final hidden state can be used for classification,
* what PyTorch autograd does in this manually implemented RNN,
* one limitation of vanilla RNNs.



# Task 4 — LSTM vs GRU Experiment (3 pts)

Train and compare an **LSTM** and a **GRU** on the same text classification task.

This task is intentionally experimental. The goal is not to get state-of-the-art accuracy. The goal is to compare two gated recurrent models under fair conditions.

## Main experiment

Use the same dataset, vocabulary, preprocessing, pretrained embedding matrix, train / validation split, batch size, sequence length, and optimizer settings for both models.

Train:


1. **Model A:** LSTM text classifier
2. **Model B:** GRU text classifier

Both models should use:

* `nn.Embedding`, preferably initialized with the same pretrained embedding matrix,
* one recurrent layer,
* final hidden state for classification,
* a final linear layer.

## Requirements


1. Implement a reusable classifier class that can switch between `lstm` and `gru`, or implement two separate classes.
2. Use the same hyperparameters for both models where possible:

```python
embedding_dim = 50
hidden_dim = 128
num_layers = 1
batch_size = 64
epochs = 3
learning_rate = 1e-3
```


3. Train both models for the same number of epochs.
4. For each model, report:
   * number of trainable parameters,
   * final training loss,
   * final validation loss,
   * final validation accuracy,
   * training time per epoch or total training time.
5. Plot validation accuracy for LSTM and GRU on the same figure.
6. Create a comparison table:

| Model | Trainable parameters | Final val loss | Final val accuracy | Training time | Comment |
|-------|---------------------:|---------------:|-------------------:|--------------:|---------|
| LSTM  | ...                  | ...            | ...                | ...           | ...     |
| GRU   | ...                  | ...            | ...                | ...           | ...     |

## Additional analysis: sequence length stress test

Choose **one** of the following mini-experiments:

### Option A — Short vs long sequences

Train or evaluate both models using two maximum sequence lengths, for example:

```python
max_len = 20
max_len = 80
```

Compare whether LSTM or GRU benefits more from longer context.

### Option B — Frozen vs trainable embeddings

For both LSTM and GRU, compare:


1. pretrained embeddings frozen,
2. pretrained embeddings trainable.

This produces four runs:

```text
LSTM frozen embeddings
LSTM trainable embeddings
GRU frozen embeddings
GRU trainable embeddings
```

### Option C — Bidirectional model

Compare one-directional and bidirectional versions of either LSTM or GRU.

Explain why the final classifier input dimension changes when the recurrent model is bidirectional.

## Required discussion

Briefly answer:


1. Which model achieved better validation accuracy?
2. Which model had fewer parameters?
3. Which model trained faster?
4. Did the result match the theory that GRU usually has fewer parameters than LSTM?


---

# Final Questions

At the end of the notebook, answer briefly:


1. What is the difference between one-hot vectors and dense word embeddings?
2. What does Word2Vec try to predict?
3. Why is full softmax expensive for large vocabularies?
4. What is the hidden state in an RNN?
5. Why do LSTM and GRU usually work better than vanilla RNNs on longer sequences?
6. What is the main architectural difference between LSTM and GRU?


---


---


# Starter Code

## General setup

```python
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
```


---

## Task 1 starter: pretrained GloVe with gensim

```python
# !pip install gensim -q

import gensim.downloader as api

vectors = api.load("glove-wiki-gigaword-50")

print("Vector size:", vectors.vector_size)
print("Vector for 'king':", vectors["king"].shape)

# Cosine similarity
print(vectors.similarity("cat", "dog"))
print(vectors.similarity("cat", "car"))

# Analogy example
result = vectors.most_similar(
    positive=["king", "woman"],
    negative=["man"],
    topn=5,
)
print(result)
```


---

## Task 3 starter: simple text dataset fallback

Use this fallback if TorchText is difficult to install.

```python
texts = [
    "team wins the football match",
    "government passes new law",
    "new smartphone has fast processor",
    "stock market rises today",
    "player scores a goal",
    "election results are announced",
    "software update improves performance",
    "company reports higher profit",
]

labels = [0, 1, 2, 3, 0, 1, 2, 3]
class_names = ["sports", "politics", "technology", "business"]
```

For a better experiment, replace this toy dataset with a subset of `AG_NEWS`.


---

## Task 3 starter: vocabulary and padding

```python
def tokenize(text):
    return text.lower().split()

special_tokens = ["<pad>", "<unk>"]
all_tokens = []
for text in texts:
    all_tokens.extend(tokenize(text))

vocab = special_tokens + sorted(set(all_tokens))
word_to_idx = {word: i for i, word in enumerate(vocab)}

pad_idx = word_to_idx["<pad>"]
unk_idx = word_to_idx["<unk>"]

max_len = 8

def encode(text, max_len=max_len):
    ids = [word_to_idx.get(tok, unk_idx) for tok in tokenize(text)]
    ids = ids[:max_len]
    ids = ids + [pad_idx] * (max_len - len(ids))
    return ids

X = torch.tensor([encode(text) for text in texts], dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)

print(X.shape, y.shape)
```


---

## Task 3 starter: manual RNN classifier

```python
class ManualRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding( ___TODO___ )
        self.hidden_dim = hidden_dim

        self.W_xh = nn.Parameter(torch.randn( ___TODO___ ) * 0.01)
        self.W_hh = nn.Parameter(torch.randn( ___TODO___ ) * 0.01)
        self.b_h = nn.Parameter(torch.zeros( ___TODO___ ))

        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x)
        # embedded shape: (batch_size, seq_len, embedding_dim)

        batch_size, seq_len, embedding_dim = embedded.shape
        h = torch.zeros(batch_size, self.hidden_dim, device=x.device)

        for t in range(seq_len):
            x_t = embedded[:, t, :]
            h = torch.tanh( ___TODO___ )

        logits = self.classifier(h)
        return logits
```

```python
model = ManualRNNClassifier(
    vocab_size=len(vocab),
    embedding_dim=50,
    hidden_dim=64,
    num_classes=4,
    padding_idx=pad_idx,
).to(device)

batch = X.to(device)
logits = model(batch)
print("input shape:", batch.shape)
print("logits shape:", logits.shape)
```


---

## Task 4 starter: LSTM / GRU classifier

```python
class RecurrentClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes,
        model_type="lstm",
        num_layers=1,
        bidirectional=False,
        dropout=0.2,
        padding_idx=0,
        pretrained_embedding_matrix=None,
        freeze_embeddings=False,
    ):
        super().__init__()

        self.model_type = model_type.lower()
        self.hidden_dim = hidden_dim
        self.bidirectional = bidirectional
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)

        if pretrained_embedding_matrix is not None:
            self.embedding.weight.data.copy_(pretrained_embedding_matrix)

        self.embedding.weight.requires_grad =  ___TODO___ 

        recurrent_dropout = dropout if num_layers > 1 else 0.0

        if self.model_type == "lstm":
            self.recurrent = nn.LSTM(
                input_size= ___TODO___ ,
                hidden_size= ___TODO___ ,
                num_layers= ___TODO___ ,
                batch_first=True,
                dropout= ___TODO___ ,
                bidirectional= ___TODO___ ,
            )
        elif self.model_type == "gru":
            self.recurrent = nn.GRU(
                input_size= ___TODO___ ,
                hidden_size= ___TODO___ ,
                num_layers= ___TODO___ ,
                batch_first=True,
                dropout= ___TODO___ ,
                bidirectional= ___TODO___ ,
            )
        else:
            raise ValueError("model_type must be 'lstm' or 'gru'")

        direction_factor = 2 if bidirectional else 1
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * direction_factor, num_classes),
        )

    def forward(self, x):
        embedded = self.embedding(x)

        if self.model_type == "lstm":
            output, (h_n, c_n) = self.recurrent(embedded)
        else:
            output, h_n = self.recurrent(embedded)

        final_output = output[:, -1, :]
        logits = self.classifier(final_output)
        return logits
```

```python
def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

lstm_model = RecurrentClassifier(
    vocab_size=len(vocab),
    embedding_dim=50,
    hidden_dim=128,
    num_classes=4,
    model_type="lstm",
    padding_idx=pad_idx,
).to(device)

gru_model = RecurrentClassifier(
    vocab_size=len(vocab),
    embedding_dim=50,
    hidden_dim=128,
    num_classes=4,
    model_type="gru",
    padding_idx=pad_idx,
).to(device)

print("LSTM trainable parameters:", count_trainable_parameters(lstm_model))
print("GRU trainable parameters:", count_trainable_parameters(gru_model))
```


---

# ✅ Deliverables

Your notebook should contain:

* pretrained embedding exploration and word algebra,
* manual full softmax calculation for Skip-gram Word2Vec,
* custom vanilla RNN forward pass using PyTorch autograd,
* LSTM vs GRU comparison,
* shape inspections,
* at least one result table,
* at least one plot,
* short written discussion for every task,
* final answers to the final questions.